# Othello - Co-occurrence Matrix

Notebook de demo pour calculer et visualiser une matrice de co-occurrence symbolique.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from boardGPT.datasets import (
    load_symbolic_pickle,
    compute_cooccurrence_matrix,
    top_cooccurring_pairs,
    plot_grouped_cooccurrence_matrix,
)

In [ ]:
# Ajuste selon ton dataset
dataset_root = Path('data/othello/othello-synthetic')
split = 'train'
max_files = 3  # augmente pour un calcul plus complet

bin_files = sorted((dataset_root / split).glob('*.bin'))[:max_files]
if not bin_files:
    raise RuntimeError(f'Aucun fichier .bin trouve dans {(dataset_root / split)}')

sequences = []
for path in bin_files:
    chunk = load_symbolic_pickle(path)
    sequences.extend(chunk)

print('files:', len(bin_files))
print('sequences:', len(sequences))
print('sample sequence length:', len(sequences[0]) if sequences else 0)

In [ ]:
# Mode 1: co-occurrence globale dans la meme sequence
matrix_same, labels = compute_cooccurrence_matrix(
    sequences=sequences,
    mode='same_sequence',
    include_self=False,
    symmetric=True,
)

In [ ]:
matrix_same = matrix_same / len(sequences)

In [ ]:
matrix_same += np.identity(matrix_same.shape[0])

In [ ]:
fig, ax = plt.subplots(figsize=(20, 19))
plot_grouped_cooccurrence_matrix(
    matrix_same,
    labels,
    title='Othello co-occurrence - same sequence',
    ax=ax,
)
plt.show()

In [ ]:
# Mode 2: co-occurrence locale (adjacent)
# Les tokens co-occurent si leur distance <= max_distance
max_distance = 2

matrix_adj, labels = compute_cooccurrence_matrix(
    sequences=sequences,
    mode='adjacent',
    max_distance=max_distance,
    include_self=False,
    symmetric=True,
)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 14))
plot_grouped_cooccurrence_matrix(
    matrix_adj,
    labels,
    title=f'Othello co-occurrence - adjacent (distance <= {max_distance})',
    ax=ax,
)
plt.show()

In [ ]:
top_pairs = top_cooccurring_pairs(matrix_adj, labels, top_k=20)
top_pairs[:10]

In [ ]:
matrix_adj

In [ ]:
from numpy.linalg import eigh

M = matrix_adj
n = M.shape[0]

P = np.eye(n) - np.ones((n,n))/n
M_centered = P @ M @ P

eigvals, eigvecs = eigh(M_centered)

In [ ]:
eigvals

In [ ]:
eigvecs.shape

In [ ]:
coords = eigvecs[:, -4:-2]
coords

In [ ]:
n_pos = np.sum(coords >= 0)
n_neg = np.sum(coords < 0)

In [ ]:
coords.shape

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(coords[:, 0], coords[:, 1])
plt.show()

## Vue interactive board -> co-occurrence

Survole une case du board a gauche pour afficher ses co-occurrences sur le board a droite.

- Distance utilisee: celle de la sequence (mode `adjacent`)
- Matrice symetrique
- La diagonale est masquee pour mieux voir les autres cases

In [ ]:
import re
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# On reutilise `matrix_adj` et `labels` calcules plus haut.
valid_square = re.compile(r'^[a-h][1-8]$')
square_labels = [s for s in labels if isinstance(s, str) and valid_square.match(s)]
if len(square_labels) == 0:
    raise RuntimeError('Aucun token de type case Othello (a1..h8) dans labels')

label_to_idx = {lab: i for i, lab in enumerate(labels)}
files = 'abcdefgh'
ranks = '12345678'

def square_to_rc(sq):
    c = files.index(sq[0])
    r = int(sq[1]) - 1
    return r, c

board_squares = [f'{f}{r}' for r in ranks for f in files]
board_mask = [1 if sq in label_to_idx else 0 for sq in board_squares]

fig_board = go.FigureWidget(
    data=[
        go.Scatter(
            x=[sq[0] for sq in board_squares],
            y=[sq[1] for sq in board_squares],
            mode='markers+text',
            text=board_squares,
            textposition='middle center',
            marker=dict(
                size=42,
                color=board_mask,
                colorscale=[[0, '#d9d9d9'], [1, '#2e8b57']],
                line=dict(color='black', width=1),
                showscale=False,
            ),
            hovertemplate='%{text}<extra></extra>',
        )
    ]
)
fig_board.update_layout(
    title='Board Othello (hover)',
    width=700,
    height=700,
    xaxis=dict(categoryorder='array', categoryarray=list(files)),
    yaxis=dict(categoryorder='array', categoryarray=list(ranks), autorange='reversed'),
)

z0 = np.full((8, 8), np.nan)
fig_heat = go.FigureWidget(
    data=[
        go.Heatmap(
            z=z0,
            x=list(files),
            y=list(ranks),
            colorscale='YlOrRd',
            zmin=0,
            zmax=float(np.nanmax(matrix_adj)),
            colorbar=dict(title='co-occ'),
            hovertemplate='case %{x}%{y}<br>value=%{z:.0f}<extra></extra>',
        )
    ]
)
fig_heat.update_layout(
    title='Survole une case a gauche',
    width=700,
    height=700,
    yaxis=dict(autorange='reversed'),
)

info = widgets.HTML('<b>Case:</b> -')

def set_heat_for_square(square):
    z = np.full((8, 8), np.nan, dtype=float)
    if square not in label_to_idx:
        fig_heat.layout.title = f'{square} (hors vocab)'
        info.value = f'<b>Case:</b> {square} | <b>Status:</b> hors vocab'
        with fig_heat.batch_update():
            fig_heat.data[0].z = z
        return

    i = label_to_idx[square]
    row = matrix_adj[i].astype(float)
    for target in board_squares:
        if target not in label_to_idx:
            continue
        rr, cc = square_to_rc(target)
        z[rr, cc] = row[label_to_idx[target]]

    rr0, cc0 = square_to_rc(square)
    z[rr0, cc0] = np.nan  # masque la diagonale

    fig_heat.layout.title = f'Co-occurrence depuis {square} (distance <= {max_distance})'
    info.value = f'<b>Case:</b> {square} | <b>Max:</b> {np.nanmax(z):.0f}'
    with fig_heat.batch_update():
        fig_heat.data[0].z = z

def _on_hover(trace, points, state):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    square = trace.text[idx]
    set_heat_for_square(square)

fig_board.data[0].on_hover(_on_hover)

display(widgets.HBox([fig_board, fig_heat]))
display(info)

# selection initiale
set_heat_for_square('d3' if 'd3' in label_to_idx else board_squares[0])